# Generate MASSVIS-EFD

This code takes as input the original MASSVIS dataset and produces MASSVIS-EFD, a multi-duration dataset composed of MASSVIS fixation points bucketed by time and bounding boxes.

## Usage

Fill in the necessary variables in the cell below and run the notebook 

In [1]:
# TODO: fill in these variables 
splits = ['train', 'val'] # include only the splits you would like to process 

bp = "/path/to/Massvis/images" # change base path to the directory where you have downloaded the MASSVIS data
annots = "/path/to/Massvis/elementLabels"
savedir = "/path/to/Massvis/GT_EFD_500_2000_5000" # change to the directory where you would like to save the multi-duration data 

## Processing Code 

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
import pandas as pd
from glob import glob
import csv
from shapely.geometry import Point
from shapely.geometry.polygon import Polygon
from skimage.draw import polygon

In [3]:
times = [500, 2000, 5000, 10000]
img_size = (240, 320)
map_size = (480, 640)

In [4]:
def parse_element_label(element_label_path: str) -> pd.DataFrame:
    """Takes a element label path containing a list of bounding boxes
    and writes it into a dataframe.

    :param element_label_path: The file to the element label file.
    :type element_label_path: str
    :return: Parsed df. (id, description, x, y, file)
    :rtype: pd.DataFrame
    """
    file_name = os.path.basename(element_label_path)
    df = pd.read_csv(element_label_path, names=[
                     'id', 'description', 'x', 'y'])
    df['file'] = file_name
    return df

def combine_rows(df_element_labels: pd.DataFrame) -> pd.DataFrame:
    """Combines the rows by id by combining the x,y coordinates to 
    a list that represents a polygon.

    df_element_labels (id, description, x, y, file)

    :param df_element_labels: DF containing the element labels.
    :type df_element_labels: pd.DataFrame
    :return: New sequashed df (id, desc, file, polygon)
    :rtype: pd.DataFrame
    """
    df = []
    element_ids = df_element_labels['id'].unique()
    for id in element_ids:
        rows = df_element_labels[df_element_labels['id'] == id]
        f = rows.values[0]
        df.append(
            {"id": f[0], "desc": f[1], "file": f[-1], 'polygon': []})
        for _, id, _, x, y, _ in rows.itertuples():
            df[-1]['polygon'].append((x, y))
    return pd.DataFrame.from_records(df)

In [5]:
def polygon_area(polygon):
    """
    compute polygon area
    polygon: list with shape [n, 2], n is the number of polygon points
    """
    area = 0
    q = polygon[-1]
    for p in polygon:
        area += p[0] * q[1] - p[1] * q[0]
        q = p
    return abs(area / 2.0)


class BBox:
    name = ''
    coords = None
    id = -1
    fixations = 0
    area = 0
    EFD = 0

    def __init__(self, name, coords, id):
        self.name = name
        self.coords = coords
        self.id = id
        self.area = polygon_area(coords)
 
    def add_fix(self):
        self.fixations += 1
        self.EFD = self.fixations / self.area

def make_efdmap(w, h, coords, imname):
    df = parse_element_label(os.path.join(annots, imname))
    element_labels = combine_rows(df)
    efdmap = np.zeros((w, h))

    boxes = []

    for id, row in element_labels.iterrows():
        #row[0] id
        #row[1] desc
        #row[2] file
        #row[3] coordinates
        boxes.append(BBox(row[1], np.array(row[3]), id))


    for c in coords: 
        x,y = int(c[0]),int(c[1])
        point = Point(x, y)
        for box in boxes:
            polyg = Polygon(box.coords)
            if polyg.contains(point):
                box.add_fix()

    for box in boxes:
        rr, cc = polygon(
            np.array(box.coords)[:, 0],
            np.array(box.coords)[:, 1],
            (w, h))
        efdmap[rr, cc] += box.EFD

    efdmap = 255*efdmap/float(np.max(efdmap))

    heatmap_img = Image.fromarray(np.uint8(np.transpose(efdmap)), "L")

    return heatmap_img

In [6]:
os.makedirs(savedir, exist_ok=True)

for cursplit in splits:
    curdir = os.path.join(bp, 'fixations_split', cursplit, 'fixationsByVis')
    os.makedirs(os.path.join(savedir, cursplit), exist_ok=True)
    origimdir = os.path.join(bp, 'targets')

    raw_img_dir = os.path.join(savedir, cursplit, 'raw_img')
    os.makedirs(raw_img_dir, exist_ok=True)
    heatmaps_dir = os.path.join(savedir, cursplit, 'heatmaps')
    os.makedirs(heatmaps_dir, exist_ok=True)
    heatmaps_accum_dir = os.path.join(savedir, cursplit, 'heatmaps_accum')
    os.makedirs(heatmaps_accum_dir, exist_ok=True)

    for time in times:
        os.makedirs(os.path.join(heatmaps_dir,'%d'%time), exist_ok=True)
        os.makedirs(os.path.join(heatmaps_accum_dir,'%d'%time), exist_ok=True)

    count = 0
    prog_step = 10

    for curfile in os.listdir(curdir):

        if (count % prog_step) == 0: 
                print("Prepping image %d" % count)
        count += 1

        basepath = os.path.basename(curfile)
        imname, ext = os.path.splitext(basepath)


        # copy the raw image
        desired_path = os.path.join(raw_img_dir, imname+'.png')
        with Image.open(os.path.join(origimdir,curfile+'.png')) as im:
            w, h = im.size

            im.save(desired_path)
            # save image in npy format
            # npy_img = preprocess_image(im, img_size[0], img_size[1])
            # np.save(os.path.join(raw_img_dir, imname + '.npy'), npy_img)

        allfiles = glob(os.path.join(curdir, curfile, 'enc' ,'*.csv'))
        for t in range(len(times)):

            allcoords = []
            allcoords_accum = []
            for subcsv in allfiles:
                subcsvfile = open(subcsv)
                csv_reader = csv.reader(subcsvfile,delimiter=',')
                timestamp = 0

                if t==0:
                    mintime = 0
                else:
                    mintime = times[t-1]
                for row in csv_reader:
                    timestamp += int(float(row[3]))
                    if mintime<=timestamp<times[t]:
                        coords = [[int(float(row[1])),int(float(row[2]))]]
                        allcoords.extend(coords)

                    if timestamp<times[t]:
                        coords = [[int(float(row[1])),int(float(row[2]))]]
                        allcoords_accum.extend(coords)

            efdm = make_efdmap(w, h, allcoords, imname)
            efdm.save(os.path.join(heatmaps_dir,str(times[t]), imname + '.png'))
            
            cum_efdm = make_efdmap(w, h, allcoords_accum, imname)
            cum_efdm.save(os.path.join(heatmaps_accum_dir,str(times[t]), imname + '.png'))


Prepping image 0
Prepping image 10
Prepping image 20
Prepping image 30
Prepping image 40
Prepping image 50
Prepping image 60
Prepping image 70
Prepping image 80
Prepping image 90
Prepping image 100
Prepping image 110
Prepping image 120
Prepping image 130
Prepping image 140
Prepping image 150
Prepping image 160
Prepping image 170
Prepping image 180
Prepping image 190
Prepping image 200
Prepping image 210
Prepping image 220
Prepping image 230
Prepping image 240
Prepping image 250
Prepping image 260
Prepping image 270
Prepping image 280
Prepping image 290
Prepping image 300
Prepping image 310
Prepping image 320
Prepping image 0
Prepping image 10
Prepping image 20
Prepping image 30
Prepping image 40
Prepping image 50
Prepping image 60
